<a href="https://colab.research.google.com/github/lucaspereira25-lps/LucasPereira-CalculoNum-rico/blob/main/Tarefa_Interpolacao_Erros_Lucas_Pereira.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

**Aluno: Lucas Pereira da Silva**  
**RA: 187566**



# Tarefa investigativa: erros em métodos de interpolação

### Aplicação: reconstrução da temperatura em uma barra aquecida

Uma barra homogênea de comprimento $L=6$ m está sujeita a temperaturas prescritas nas extremidades e recebe calor de uma fonte distribuída em seu interior. Sensores medem a temperatura em posições discretas. A equipe precisa estimar a temperatura em posições onde não há sensor e avaliar a incerteza dessas estimativas.

Esta atividade combina um modelo simplificado de transferência de calor com interpolação numérica. O objetivo não é resolver a equação diferencial de calor por diferenças finitas. O perfil analítico será usado como referência, para que os erros de diferentes métodos de interpolação possam ser medidos e comparados.

Você vai comparar **polinômio global de Lagrange** e uma **spline cúbica**, primeiro com dados exatos e depois com medições perturbadas por ruído.


## 1. Problema físico e equação de transferência de calor

### 1.1 Modelo físico

Considere uma barra sólida homogênea, com comprimento $L=6$ m e seção transversal constante. A temperatura varia principalmente ao longo do eixo $x$. As extremidades são mantidas a temperaturas conhecidas. No interior, uma fonte térmica volumétrica distribui energia ao longo da barra.

A equação geral de energia térmica, para propriedades constantes e sem mudança de fase, pode ser escrita como

$$\rho c_p\frac{\partial T}{\partial t}=k\nabla^2T+\dot q^{\prime\prime\prime}-\rho c_p\,\mathbf{u}\cdot\nabla T,$$

em que $T$ é a temperatura (K ou °C para diferenças), $t$ é o tempo (s), $\rho$ é a massa específica (kg m$^{-3}$), $c_p$ é o calor específico (J kg$^{-1}$ K$^{-1}$), $k$ é a condutividade térmica (W m$^{-1}$ K$^{-1}$), $\dot q^{\prime\prime\prime}$ é a geração volumétrica de calor (W m$^{-3}$) e $\mathbf{u}$ representa o movimento de um fluido, quando há advecção.

Para esta atividade, adotamos uma **barra sólida em regime estacionário**, sem escoamento interno, com condução unidimensional e geração de calor. Assim, $\partial T/\partial t=0$, $\mathbf{u}=0$ e $\nabla^2T=d^2T/dx^2$. A equação fica

$$k\frac{d^2T}{dx^2}+\dot q^{\prime\prime\prime}(x)=0,\qquad 0<x<L,$$

com condições de contorno de Dirichlet

$$T(0)=18\ ^\circ\mathrm C,\qquad T(L)=20.4\ ^\circ\mathrm C.$$

O primeiro termo representa a condução: diferenças espaciais de temperatura geram fluxo de calor. O termo $\dot q^{\prime\prime\prime}(x)$ representa energia adicionada por unidade de volume e de tempo. O sinal positivo de $\dot q^{\prime\prime\prime}$ indica geração interna de calor.

### 1.2 Perfil de referência usado no experimento

Para criar um caso com solução conhecida, definimos $\xi=x/L$ e adotamos

$$T(x)=18+2.4\,\xi+2.5\sin(\pi\xi),\qquad \xi=\frac{x}{L},\quad 0\le x\le L.$$

A parcela linear garante as temperaturas prescritas nas extremidades. A parcela senoidal representa um aquecimento distribuído mais intenso no interior e nulo nas bordas. Derivando duas vezes em relação à coordenada física $x$,

$$\frac{d^2T}{dx^2}=-\frac{2.5\pi^2}{L^2}\sin\left(\frac{\pi x}{L}\right).$$

Portanto, esse perfil satisfaz a equação estacionária se a geração volumétrica for

$$\dot q^{\prime\prime\prime}(x)=\frac{2.5\,k\pi^2}{L^2}\sin\left(\frac{\pi x}{L}\right).$$

Como $\sin(\pi x/L)\ge0$ dentro da barra, a fonte é não negativa. Não precisamos escolher um valor numérico para $k$: o objetivo é verificar a coerência do perfil e estudar a interpolação. A função analítica é uma referência idealizada; em uma aplicação experimental, seria preciso estimar propriedades, fontes, perdas e condições de contorno.

### 1.3 Questão investigativa

**Como a resposta depende da quantidade e posição dos sensores e do ruído de medição?**



### O que significa erro de interpolação?

Em um ponto de avaliação $x$, o erro assinado é $e(x)=p(x)-T(x)$. O erro absoluto é $|e(x)|$. Como a interpolação reproduz os nós, o erro nesses pontos pode ser praticamente zero; por isso, avaliaremos também muitos pontos **entre** os nós.

Usaremos: erro máximo absoluto $E_{max}=\max_j|e_j|$; erro médio absoluto (MAE) $=\frac{1}{m}\sum_j|e_j|$; raiz do erro quadrático médio (RMSE) $=\sqrt{\frac{1}{m}\sum_j e_j^2}$. O RMSE penaliza mais erros grandes.

## 2. Figura do problema e preparação do ambiente

A figura esquematiza a barra, os sensores, as temperaturas nas extremidades e a fonte térmica interna. A curva corresponde ao perfil de referência usado no experimento. Os pontos são as posições onde os sensores amostram a temperatura.

In [ ]:
# Preparação do modelo e bibliotecas
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy.interpolate import BarycentricInterpolator, interp1d, CubicSpline
from matplotlib.patches import Rectangle

np.set_printoptions(precision=4, suppress=True)
plt.rcParams["figure.figsize"] = (9, 5)
L = 6.0  # comprimento da barra, m

def temperatura_real(x):
    """Perfil analítico de referência em °C; x é dado em metros."""
    x = np.asarray(x, dtype=float)
    xi = x/L
    return 18 + 2.4*xi + 2.5*np.sin(np.pi*xi)

x_min, x_max = 0.0, L
x_fino = np.linspace(x_min, x_max, 2001)
T_fino = temperatura_real(x_fino)

# Figura do problema: esquema físico e perfil analítico em painéis separados

x_sensores_fig = np.linspace(0, L, 7)
T_sensores_fig = temperatura_real(x_sensores_fig)

fig = plt.figure(figsize=(13, 8), constrained_layout=True)
gs = fig.add_gridspec(2, 1, height_ratios=[0.85, 1.5])
ax_esq = fig.add_subplot(gs[0])
ax_perfil = fig.add_subplot(gs[1])

# Painel (a): barra física, sensores e fonte interna distribuída
ax_esq.set_title("(a) Modelo físico: barra sólida com geração interna de calor", loc="left", fontsize=12, weight="bold")
barra_y, barra_h = 0.40, 0.25
ax_esq.add_patch(Rectangle((0, barra_y), L, barra_h, facecolor="#e8edf2",
                           edgecolor="#34495e", linewidth=1.6, zorder=1))
# Sensores ao longo do eixo da barra
ax_esq.scatter(x_sensores_fig, np.full_like(x_sensores_fig, barra_y+barra_h/2),
               s=65, marker="o", facecolor="#1769aa", edgecolor="white", linewidth=0.9,
               zorder=4, label="Sensores")
# Aquecimento interno: setas iguais identificam geração distribuída, sem sugerir fluxo de calor externo
for xpos in np.linspace(0.45, 5.55, 11):
    ax_esq.annotate("", xy=(xpos, barra_y-0.015), xytext=(xpos, 0.08),
                    arrowprops={"arrowstyle":"-|>", "color":"#d97706", "lw":1.5}, zorder=3)
ax_esq.text(L/2, 0.015, r"Geração volumétrica distribuída $\dot{q}_{\mathrm{gen}}(x)$",
            ha="center", va="bottom", color="#9a4d00", fontsize=10,
            bbox={"facecolor":"white", "edgecolor":"none", "alpha":0.88, "pad":2.5})
ax_esq.text(0, 0.77, r"$T(0)=18{,}0\ ^\circ\mathrm{C}$", ha="left", va="bottom", fontsize=10)
ax_esq.text(L, 0.77, r"$T(L)=20{,}4\ ^\circ\mathrm{C}$", ha="right", va="bottom", fontsize=10)
# Linha de dimensão e eixo longitudinal
ax_esq.annotate("", xy=(L, 1.00), xytext=(0, 1.00),
                arrowprops={"arrowstyle":"<->", "color":"#34495e", "lw":1.2})
ax_esq.text(L/2, 1.025, r"Comprimento $L=6\,\mathrm{m}$", ha="center", va="bottom", fontsize=9)
for i, xpos in enumerate(x_sensores_fig):
    ax_esq.text(xpos, barra_y+barra_h+0.04, rf"$x_{i}$", ha="center", va="bottom", fontsize=8, color="#174a73")
ax_esq.set_xlim(-0.15, L+0.15); ax_esq.set_ylim(0, 1.2)
ax_esq.set_xlabel("Coordenada ao longo da barra, x (m)")
ax_esq.set_yticks([]); ax_esq.grid(False)
for spine in ["left", "right", "top"]: ax_esq.spines[spine].set_visible(False)

# Painel (b): curva de referência e posições amostradas
ax_perfil.set_title("(b) Perfil de temperatura usado como referência para avaliar a interpolação",
                    loc="left", fontsize=12, weight="bold")
ax_perfil.plot(x_fino, T_fino, color="#b23a48", linewidth=2.5,
               label=r"Perfil analítico $T(x)$", zorder=2)
ax_perfil.scatter(x_sensores_fig, T_sensores_fig, s=62, marker="o",
                  facecolor="#1769aa", edgecolor="white", linewidth=0.9,
                  label="Leituras nos sensores (caso sem ruído)", zorder=4)
ax_perfil.fill_between(x_fino, T_fino, np.min(T_fino)-0.3,
                       color="#b23a48", alpha=0.08, zorder=1)
ax_perfil.annotate(r"$T(0)=18{,}0\ ^\circ\mathrm{C}$",
                   xy=(0, temperatura_real(0)), xytext=(0.35, 18.35),
                   arrowprops={"arrowstyle":"->", "color":"#555"}, fontsize=9)
ax_perfil.annotate(r"$T(L)=20{,}4\ ^\circ\mathrm{C}$",
                   xy=(L, temperatura_real(L)), xytext=(4.6, 20.75),
                   arrowprops={"arrowstyle":"->", "color":"#555"}, fontsize=9)
ax_perfil.set_xlim(0, L); ax_perfil.set_ylim(17.7, 22.5)
ax_perfil.set_xlabel("Posição x (m)")
ax_perfil.set_ylabel("Temperatura T(x) (°C)")
ax_perfil.grid(True, alpha=0.25); ax_perfil.legend(loc="lower center", ncol=2, frameon=True)
fig.suptitle("Problema de transferência de calor e reconstrução por interpolação", fontsize=14, weight="bold")
plt.show()

print(f"Temperaturas nas extremidades: T(0)={temperatura_real(0):.1f} °C e T(L)={temperatura_real(L):.1f} °C")

Spline quadrática

In [ ]:
def spline_quadratica_coef(x, y):
    """Spline quadrática C1 com condição c_0=0; retorna a,b,c locais."""
    x = np.asarray(x, float); y = np.asarray(y, float)
    n = len(x)-1
    if n < 1 or len(y) != len(x) or np.any(np.diff(x) <= 0):
        raise ValueError("x deve ser crescente e ter o mesmo tamanho de y.")
    A = np.zeros((3*n, 3*n)); rhs = np.zeros(3*n); row = 0
    # S_i(x_i)=y_i
    for i in range(n):
        A[row, i] = 1; rhs[row] = y[i]; row += 1
    # S_i(x_(i+1))=y_(i+1)
    for i in range(n):
        h = x[i+1]-x[i]
        A[row, i] = 1; A[row, n+i] = h; A[row, 2*n+i] = h*h
        rhs[row] = y[i+1]; row += 1
    # Continuidade da primeira derivada nos nós internos
    for i in range(n-1):
        h = x[i+1]-x[i]
        A[row, n+i] = 1; A[row, 2*n+i] = 2*h; A[row, n+i+1] = -1
        row += 1
    # Condição de contorno: c_0=0
    A[row, 2*n] = 1
    coef = np.linalg.solve(A, rhs)
    return coef[:n], coef[n:2*n], coef[2*n:]

def spline_quadratica_avalia(x, coef, xq):
    a, b, c = coef
    xq = np.asarray(xq, dtype=float)
    idx = np.clip(np.searchsorted(x, xq, side="right")-1, 0, len(a)-1)
    t = xq-x[idx]
    return a[idx] + b[idx]*t + c[idx]*t**2

def criar_metodos(x, y):
    """Retorna dicionário: interpoladores callable para quatro métodos."""
    lagrange = BarycentricInterpolator(x, y)
    linear = interp1d(x, y, kind="linear", bounds_error=True)
    quad_coef = spline_quadratica_coef(x, y)
    quad = lambda z: spline_quadratica_avalia(x, quad_coef, z)
    cubica = CubicSpline(x, y, bc_type="natural")
    return {"Lagrange global": lagrange,
            "Spline linear": linear,
            "Spline quadrática": quad,
            "Spline cúbica natural": cubica}

def calcular_metricas(metodos, x_teste, y_verdadeiro):
    linhas = []
    for nome, metodo in metodos.items():
        erro = np.asarray(metodo(x_teste))-np.asarray(y_verdadeiro)
        linhas.append({"Método": nome,
                       "Erro máximo (°C)": np.max(np.abs(erro)),
                       "MAE (°C)": np.mean(np.abs(erro)),
                       "RMSE (°C)": np.sqrt(np.mean(erro**2))})
    return pd.DataFrame(linhas).sort_values("RMSE (°C)").reset_index(drop=True)

print("Funções de interpolação e métricas definidas.")

## 3. Métodos de interpolação e métricas de erro

O interpolador global de Lagrange passa por todos os sensores usando um único polinômio. Usamos a forma baricêntrica, que evita expandir diretamente os coeficientes e costuma ser numericamente mais estável.

As splines constroem polinômios por intervalos. A spline linear conecta pontos por retas. A quadrática deste notebook impõe continuidade da função e da primeira derivada e usa a condição de contorno $c_0=0$ no primeiro intervalo. A spline cúbica natural impõe continuidade até a segunda derivada e segunda derivada nula nas extremidades.

Para quantificar a diferença, calculamos $e(x)=T_{interp}(x)-T_{ref}(x)$. O erro máximo mostra o pior desvio na malha. O MAE dá o desvio absoluto médio. O RMSE penaliza mais os erros de grande magnitude. Todos têm unidade de graus Celsius.

A malha fina serve como conjunto de pontos de avaliação: os valores nela não são novos sensores. Assim, a comparação mede desempenho entre os nós.

In [ ]:
def spline_quadratica_coef(x, y):
    """Spline quadrática C1 com condição c_0=0; retorna a,b,c locais."""
    x = np.asarray(x, float); y = np.asarray(y, float)
    n = len(x)-1
    if n < 1 or len(y) != len(x) or np.any(np.diff(x) <= 0):
        raise ValueError("x deve ser crescente e ter o mesmo tamanho de y.")
    A = np.zeros((3*n, 3*n)); rhs = np.zeros(3*n); row = 0
    # S_i(x_i)=y_i
    for i in range(n):
        A[row, i] = 1; rhs[row] = y[i]; row += 1
    # S_i(x_(i+1))=y_(i+1)
    for i in range(n):
        h = x[i+1]-x[i]
        A[row, i] = 1; A[row, n+i] = h; A[row, 2*n+i] = h*h
        rhs[row] = y[i+1]; row += 1
    # Continuidade da primeira derivada nos nós internos
    for i in range(n-1):
        h = x[i+1]-x[i]
        A[row, n+i] = 1; A[row, 2*n+i] = 2*h; A[row, n+i+1] = -1
        row += 1
    # Condição de contorno: c_0=0
    A[row, 2*n] = 1
    coef = np.linalg.solve(A, rhs)
    return coef[:n], coef[n:2*n], coef[2*n:]

def spline_quadratica_avalia(x, coef, xq):
    a, b, c = coef
    xq = np.asarray(xq, dtype=float)
    idx = np.clip(np.searchsorted(x, xq, side="right")-1, 0, len(a)-1)
    t = xq-x[idx]
    return a[idx] + b[idx]*t + c[idx]*t**2

def criar_metodos(x, y):
    """Retorna dicionário: interpoladores callable para quatro métodos."""
    lagrange = BarycentricInterpolator(x, y)
    linear = interp1d(x, y, kind="linear", bounds_error=True)
    quad_coef = spline_quadratica_coef(x, y)
    quad = lambda z: spline_quadratica_avalia(x, quad_coef, z)
    cubica = CubicSpline(x, y, bc_type="natural")
    return {"Lagrange global": lagrange,
            "Spline linear": linear,
            "Spline quadrática": quad,
            "Spline cúbica natural": cubica}

def calcular_metricas(metodos, x_teste, y_verdadeiro):
    linhas = []
    for nome, metodo in metodos.items():
        erro = np.asarray(metodo(x_teste))-np.asarray(y_verdadeiro)
        linhas.append({"Método": nome,
                       "Erro máximo (°C)": np.max(np.abs(erro)),
                       "MAE (°C)": np.mean(np.abs(erro)),
                       "RMSE (°C)": np.sqrt(np.mean(erro**2))})
    return pd.DataFrame(linhas).sort_values("RMSE (°C)").reset_index(drop=True)

print("Funções de interpolação e métricas definidas.")

## Etapa 1 — Interpolação com dados exatos
Suponha primeiro que os sensores sejam perfeitos. Coloque sete sensores em posições igualmente espaçadas, incluindo as duas extremidades. Os valores registrados são então exatamente $T(x_i)$.

### Passo a passo
1. Construa o vetor das posições e avalie o perfil de referência nesses pontos.
2. Crie a interpolação com `criar_metodos`.
3. Avalie cada método em `x_fino`, uma malha densa entre 0 e 6 m.
4. Calcule $E_{max}$, MAE e RMSE comparando cada curva com `T_fino`.
5. Faça um gráfico das temperaturas e outro dos erros assinados.




In [ ]:
x_nos_7 = np.linspace(x_min, x_max, 7)
y_nos_7 = temperatura_real(x_nos_7)
metodos_7 = criar_metodos(x_nos_7, y_nos_7)

metricas_7 = calcular_metricas(metodos_7, x_fino, T_fino)
display(metricas_7.round(6))

plt.figure(figsize=(10, 5))
plt.plot(x_fino, T_fino, color="black", lw=2.5, label="Referência analítica")
for nome, metodo in metodos_7.items():
    plt.plot(x_fino, metodo(x_fino), lw=1.5, label=nome)
plt.scatter(x_nos_7, y_nos_7, color="black", s=35, zorder=5, label="Sensores exatos")
plt.xlabel("Posição x (m)"); plt.ylabel("Temperatura (°C)")
plt.title("Etapa 1: interpolação com sete sensores exatos")
plt.grid(alpha=.25); plt.legend(); plt.show()


In [ ]:
plt.figure(figsize=(10, 5))
for nome, metodo in metodos_7.items():
    plt.plot(x_fino, metodo(x_fino)-T_fino, lw=1.5, label=nome)
plt.axhline(0, color="black", lw=1)
plt.xlabel("Posição x (m)"); plt.ylabel("Erro assinado (°C)")
plt.title("Etapa 1: erros entre os sensores")
plt.grid(alpha=.25); plt.legend(); plt.show()


## Etapa 2 — Número e posição dos sensores

Nesta etapa, investigue somente o polinômio global de Lagrange. A interpolação polinomial global pode oscilar entre nós, sobretudo perto das extremidades, quando se usam muitos nós equiespaçados. A distribuição de Chebyshev concentra nós nas bordas e pode reduzir esse efeito.

Para $N$ nós de Chebyshev-Lobatto em $[a,b]$, use

$$x_k=\frac{a+b}{2}+\frac{b-a}{2}\cos\left(\frac{k\pi}{N-1}\right),\quad k=0,\ldots,N-1.$$

Ordene as posições crescentemente. Compare $N=5,9,13,17$ para nós equiespaçados e de Chebyshev. Para cada caso, avalie o erro em todos os pontos da malha fina e registre o erro máximo e o RMSE.

**Atenção:** uma redução de erro neste perfil particular não prova que o mesmo ocorrerá para qualquer função. Relacione a observação à suavidade e à estrutura do perfil usado.

In [ ]:
def nos_chebyshev_lobatto(a, b, N):
    k = np.arange(N)
    z = np.cos(k*np.pi/(N-1))
    return np.sort((a+b)/2 + (b-a)*z/2)

linhas = []
for N in (5, 9, 13, 17):
    for distribuicao, nos in (("Equiespaçados", np.linspace(x_min, x_max, N)),
                              ("Chebyshev-Lobatto", nos_chebyshev_lobatto(x_min, x_max, N))):
        p = BarycentricInterpolator(nos, temperatura_real(nos))
        e = np.asarray(p(x_fino))-T_fino
        linhas.append({"N":N, "Distribuição":distribuicao,
                       "Erro máximo (°C)":np.max(np.abs(e)),
                       "RMSE (°C)":np.sqrt(np.mean(e**2))})
tabela_nos = pd.DataFrame(linhas)
display(tabela_nos.round(8))

fig, axes = plt.subplots(1, 2, figsize=(12, 4.5))
for distribuicao, marcador in (("Equiespaçados", "o"), ("Chebyshev-Lobatto", "s")):
    d=tabela_nos[tabela_nos["Distribuição"]==distribuicao]
    axes[0].plot(d["N"], d["Erro máximo (°C)"], marker=marcador, label=distribuicao)
    axes[1].plot(d["N"], d["RMSE (°C)"], marker=marcador, label=distribuicao)
for ax, titulo, ylabel in zip(axes, ("Erro máximo", "RMSE"), ("°C", "°C")):
    ax.set_title(titulo); ax.set_xlabel("Número de sensores N"); ax.set_ylabel(ylabel)
    ax.grid(alpha=.25); ax.legend()
plt.suptitle("Etapa 2: efeito da quantidade e posição dos sensores")
plt.tight_layout(); plt.show()


## Etapa 3 — Estudo de caso e decisão técnica

Faça $N$ variar de 4 a 40. Compare erro máximo e RMSE em escala logarítmica para Lagrange equiespaçado, Lagrange com Chebyshev e spline cúbica natural. Registre o tempo de construção e de avaliação. Discuta o custo e a estabilidade numérica. Para o perfil suave escolhido, haverá necessariamente um único método superior para todo $N$?

In [ ]:
import time

linhas_bench = []
for N in range(4, 41):
    configuracoes = [
        ("Lagrange equiespaçado", np.linspace(x_min, x_max, N), "lagrange"),
        ("Lagrange Chebyshev", nos_chebyshev_lobatto(x_min, x_max, N), "lagrange"),
        ("Spline cúbica natural", np.linspace(x_min, x_max, N), "spline"),
    ]
    for metodo_nome, nos, tipo in configuracoes:
        valores = temperatura_real(nos)
        t0=time.perf_counter()
        interp = BarycentricInterpolator(nos, valores) if tipo=="lagrange" else CubicSpline(nos, valores, bc_type="natural")
        t_construcao=time.perf_counter()-t0
        t0=time.perf_counter(); estimativa=interp(x_fino); t_avaliacao=time.perf_counter()-t0
        erro=np.asarray(estimativa)-T_fino
        linhas_bench.append({"N":N,"Método":metodo_nome,"Erro máximo (°C)":np.max(np.abs(erro)),
            "RMSE (°C)":np.sqrt(np.mean(erro**2)),"Construção (ms)":1000*t_construcao,
            "Avaliação 2001 pontos (ms)":1000*t_avaliacao})
benchmark = pd.DataFrame(linhas_bench)
display(benchmark.groupby("Método").agg({"Erro máximo (°C)":"min", "RMSE (°C)":"min",
    "Construção (ms)":"median", "Avaliação 2001 pontos (ms)":"median"}).round(6))

fig, axes=plt.subplots(1,2,figsize=(12,4.5))
for nome, grupo in benchmark.groupby("Método"):
    axes[0].semilogy(grupo["N"], np.maximum(grupo["Erro máximo (°C)"],1e-16),label=nome)
    axes[1].semilogy(grupo["N"], np.maximum(grupo["RMSE (°C)"],1e-16),label=nome)
axes[0].set_title("Erro máximo"); axes[1].set_title("RMSE")
for ax in axes:
    ax.set_xlabel("Número de sensores N"); ax.set_ylabel("Erro (°C, escala log)"); ax.grid(alpha=.25,which="both"); ax.legend()
plt.suptitle("Etapa 3: precisão para N entre 4 e 40"); plt.tight_layout(); plt.show()

fig, axes=plt.subplots(1,2,figsize=(12,4.5))
for nome, grupo in benchmark.groupby("Método"):
    axes[0].plot(grupo["N"], grupo["Construção (ms)"],label=nome)
    axes[1].plot(grupo["N"], grupo["Avaliação 2001 pontos (ms)"],label=nome)
axes[0].set_title("Construção"); axes[1].set_title("Avaliação")
for ax in axes:
    ax.set_xlabel("Número de sensores N"); ax.set_ylabel("Tempo (ms)"); ax.grid(alpha=.25); ax.legend()
plt.suptitle("Etapa 3: custo computacional (tempos dependem do ambiente)"); plt.tight_layout(); plt.show()


### Experimento complementar: sensores com ruído

Para tornar a recomendação concreta, adotamos ruído gaussiano independente com desvio-padrão de 0,20 °C e semente fixa para reprodutibilidade. Este nível de ruído é uma hipótese didática, não uma especificação de sensor real. Comparamos Lagrange global e spline cúbica natural para 7 e 13 sensores; mostramos também uma suavização por spline cúbica de suavização. A escolha do parâmetro de suavização deve, em campo, ser calibrada pela incerteza instrumental ou validação cruzada.


In [ ]:
from scipy.interpolate import UnivariateSpline

rng=np.random.default_rng(187569)
sigma_ruido=0.20
linhas_ruido=[]
fig, axes=plt.subplots(1,2,figsize=(13,5),sharey=True)
for ax,N in zip(axes,(7,13)):
    xn=np.linspace(x_min,x_max,N); yn=temperatura_real(xn)+rng.normal(0,sigma_ruido,N)
    lag=BarycentricInterpolator(xn,yn); cub=CubicSpline(xn,yn,bc_type="natural")
    # s aproximadamente soma esperada dos quadrados do ruído; não interpola cada flutuação
    suav=UnivariateSpline(xn,yn,k=3,s=N*sigma_ruido**2)
    ax.plot(x_fino,T_fino,color="black",lw=2,label="Referência (desconhecida em campo)")
    ax.plot(x_fino,lag(x_fino),label="Lagrange global")
    ax.plot(x_fino,cub(x_fino),label="Spline cúbica interpolante")
    ax.plot(x_fino,suav(x_fino),label="Spline suavizante")
    ax.scatter(xn,yn,color="black",s=22,zorder=4,label="Leituras ruidosas")
    ax.set_title(f"{N} sensores, σ={sigma_ruido:.2f} °C"); ax.set_xlabel("x (m)"); ax.grid(alpha=.25)
    for nome, pred in (("Lagrange global",lag),("Spline cúbica interpolante",cub),("Spline suavizante",suav)):
        e=pred(x_fino)-T_fino
        linhas_ruido.append({"N":N,"Método":nome,"Erro máximo (°C)":np.max(np.abs(e)),
            "MAE (°C)":np.mean(np.abs(e)),"RMSE (°C)":np.sqrt(np.mean(e**2))})
axes[0].set_ylabel("Temperatura (°C)"); axes[1].legend(loc="best",fontsize=8)
plt.suptitle("Experimento complementar: interpolação e ruído de medição"); plt.tight_layout(); plt.show()
tabela_ruido=pd.DataFrame(linhas_ruido); display(tabela_ruido.round(6))


## Etapa 4 — Estudo de caso e decisão técnica

Uma equipe precisa estimar a temperatura em posições internas da barra, com sensores ruidosos. Não haverá sensores em todos os pontos e a solução verdadeira será desconhecida em campo. Com base nos experimentos, escreva uma recomendação técnica.

A recomendação deve responder:
- que método escolheria com poucos sensores exatos e por quê;
- como a recomendação muda quando se aumenta o número de sensores;
- o que faria quando os sensores apresentam ruído;
- que cuidados adotaria perto das extremidades;
- que limitações permanecem quando a temperatura verdadeira não é conhecida.

Use pelo menos duas métricas da atividade. Diferencie interpolação dentro do intervalo medido de extrapolação além dele.


### Recomendação técnica

Com poucos sensores exatos, escolheria uma spline cúbica natural como ponto de partida: ela é local por trechos, reproduz os nós e evita que um único polinômio de alto grau determine toda a curva. A decisão deve ser confirmada pelo erro máximo e pelo RMSE obtidos na Etapa 1; com sete nós equiespaçados, a tabela permite comparar diretamente os quatro métodos. Se houver pouquíssimos nós e a função for suave, Lagrange global pode ser competitivo, mas não deve ser extrapolado para muitos nós equiespaçados sem verificar oscilações.

Ao aumentar N, verificaria a tendência de **Emax** e **RMSE**, além do tempo de construção e avaliação. Nós de Chebyshev podem reduzir oscilações de borda do polinômio global; ainda assim, os resultados dependem deste perfil suave específico. Não existe garantia de um método vencedor para todo N: spline cúbica pode ser mais precisa em uma faixa, enquanto a condição de contorno natural introduz seu próprio viés nas bordas.

Com ruído, não usaria interpolação exata como regra: ela força a curva a passar por cada leitura, inclusive pelo erro instrumental, e o polinômio global pode amplificar oscilações. Preferiria uma spline suavizante ou regressão regularizada, calibrando a suavização a partir da incerteza dos sensores e validando em medições independentes. Compararia **RMSE** (erro típico com penalização de desvios grandes) e **Emax** (pior desvio observado); MAE é uma medida complementar mais robusta a poucos desvios extremos. O experimento acima usa σ=0,20 °C apenas como hipótese e mostra como calcular essas métricas, sem alegar que essa seja a precisão real dos sensores.

Inspecionaria com atenção as extremidades: o Lagrange equiespaçado pode oscilar ali, e a spline cúbica natural impõe segunda derivada zero nas pontas, condição que pode não representar a física. Incluiria sensores próximos às bordas e, se houver informação física confiável, usaria condições de contorno compatíveis.

As métricas desta atividade são possíveis porque conhecemos a solução analítica; no campo, a temperatura verdadeira não está disponível em todos os pontos. Usaria sensores de validação não empregados no ajuste, repetição de leituras, estimativa de incerteza e análise de sensibilidade a N e à posição dos sensores. Interpolação estima **dentro** do intervalo coberto pelos sensores; extrapolação além dele não é sustentada por estes erros e exige um modelo físico adicional.
